
# 任意 CSV 按经纬度匹配省级 shp，并原地覆盖原文件

这个 notebook 是按你给的参考 `ipynb` 思路改写的，功能如下：

1. 读取任意一个 CSV 文件；
2. 自动识别经度/纬度列；
3. 与省级 shp 做空间匹配；
4. 向 CSV 中回填 / 覆盖 `ADCODE99` 和 `NAME_EN_JX` 两列；
5. 对 **未匹配点** 按 **最近省份** 进行匹配；
6. 最后 **直接覆盖原 CSV 文件**（会先自动备份一个 `.bak` 文件）。

## 你只需要改哪里？

只需要改下面参数单元里的：

```python
input_csv = r"你的 csv 路径"
```

其余都可以直接运行。

## 匹配逻辑

- 先用 `within` 做严格面内匹配；
- 对 `within` 未匹配到的点，再用 `nearest` 做最近匹配；
- 这样边界点、轻微偏移点也能补上。

## 说明

- 如果 shp 中本来就有 `NAME_EN_JX` 字段，脚本直接使用；
- 如果你当前这份 shp 没有 `NAME_EN_JX`，脚本会优先根据 `ADCODE99` 自动补标准省名；
- 仍补不上的情况下，才会退回 shp 里的名称字段。


In [ ]:

import os
import re
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", category=UserWarning)

print("依赖导入完成。")
print("pandas version:", pd.__version__)
print("geopandas version:", gpd.__version__)


In [ ]:

# =========================
# 参数区：你只需要改 input_csv
# =========================
home_dir = os.path.expanduser("/path/to/home/")
base_path = os.path.join(home_dir, "PROJECT_DIR", "Papers", "4_NC")
# 这里只改成你的 CSV 路径即可
# input_csv = os.path.join(base_path, "data", "output", "Positive_Negative_balanced_25366.csv")
input_csv = os.path.join(base_path, "data", "output", "Points_China_all_10km.csv")
# 如经纬度列名很特殊、自动识别失败，可手动指定；否则保持 None
lon_col = None
lat_col = None

# 是否先备份原文件（会生成同名 .bak 文件）
backup_original = True

# 是否把匹配方式一并写回 CSV
# False：只回填 / 覆盖 ADCODE99 和 NAME_EN_JX
# True：额外写入 province_match_method 列，便于核查
save_match_method_column = False

# 输出编码：None 表示尽量沿用原 CSV 编码
# 如果你想让 Excel 打开更稳妥，也可改成 "utf-8-sig"
output_encoding = None


# =========================================
# 省级 shp 路径：优先使用你原 notebook 的路径写法
# 如果找不到，脚本会自动去 CSV 同目录 / 当前目录兜底查找
# =========================================

province_no_TW_AM_HK_shp = os.path.join(
    base_path, "data", "Administrative_divisions_of_china",
    "no_TW_AM_HK", "china_pro2_no_TW_AM_HK_geographical_division.shp"
)

print("input_csv =", input_csv)
print("province_no_TW_AM_HK_shp =", province_no_TW_AM_HK_shp)


In [ ]:

# 工具函数

PROVINCE_NAME_EN_FALLBACK = {
    110000: "Beijing",
    120000: "Tianjin",
    130000: "Hebei",
    140000: "Shanxi",
    150000: "Inner Mongolia",
    210000: "Liaoning",
    220000: "Jilin",
    230000: "Heilongjiang",
    310000: "Shanghai",
    320000: "Jiangsu",
    330000: "Zhejiang",
    340000: "Anhui",
    350000: "Fujian",
    360000: "Jiangxi",
    370000: "Shandong",
    410000: "Henan",
    420000: "Hubei",
    430000: "Hunan",
    440000: "Guangdong",
    450000: "Guangxi",
    460000: "Hainan",
    500000: "Chongqing",
    510000: "Sichuan",
    520000: "Guizhou",
    530000: "Yunnan",
    540000: "Tibet",
    610000: "Shaanxi",
    620000: "Gansu",
    630000: "Qinghai",
    640000: "Ningxia",
    650000: "Xinjiang",
}


def read_csv_with_fallback(path, **kwargs):
    """尝试多种常见编码读取 CSV。"""
    encodings = ["utf-8", "utf-8-sig", "gb18030", "gbk"]
    errors = []

    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc, **kwargs)
            return df, enc
        except Exception as e:
            errors.append(f"{enc}: {type(e).__name__}: {e}")

    try:
        df = pd.read_csv(path, encoding="latin1", **kwargs)
        print("⚠️ 采用 latin1 兜底读取，中文可能乱码，请检查原始 CSV 编码。")
        return df, "latin1"
    except Exception:
        raise RuntimeError("CSV 读取失败。已尝试编码：\n" + "\n".join(errors))


def normalize_col(name):
    return re.sub(r"[^0-9a-zA-Z\u4e00-\u9fff]+", "", str(name)).lower()


def detect_column(df, manual_col, candidates, fuzzy_tokens, label):
    """优先手动指定；否则自动识别列名。"""
    if manual_col is not None:
        if manual_col not in df.columns:
            raise KeyError(f"手动指定的 {label} 列不存在：{manual_col}")
        return manual_col

    norm_map = {normalize_col(col): col for col in df.columns}

    for cand in candidates:
        cand_norm = normalize_col(cand)
        if cand_norm in norm_map:
            return norm_map[cand_norm]

    matched = []
    for col in df.columns:
        norm = normalize_col(col)
        if any(token in norm for token in fuzzy_tokens):
            matched.append(col)

    if len(matched) == 1:
        return matched[0]

    if len(matched) > 1:
        matched = sorted(matched, key=lambda x: (len(normalize_col(x)), str(x)))
        return matched[0]

    raise KeyError(
        f"未找到 {label} 列。\n"
        f"当前 CSV 列名：{df.columns.tolist()}\n"
        f"请手动设置 {label} 列名。"
    )


def resolve_province_shp(input_csv, preferred_path=None):
    """优先使用给定路径；找不到时自动在常见位置查找 shp。"""
    candidates = []

    if preferred_path:
        candidates.append(Path(preferred_path).expanduser())

    csv_dir = Path(input_csv).expanduser().resolve().parent
    cwd = Path.cwd()

    candidate_names = [
        "china_pro2_no_TW_AM_HK_geographical_division.shp",
        "china_pro2_no_TW_AM_HK.shp",
        os.path.join(
            "data", "Administrative_divisions_of_china", "no_TW_AM_HK",
            "china_pro2_no_TW_AM_HK_geographical_division.shp"
        ),
        os.path.join(
            "data", "Administrative_divisions_of_china", "no_TW_AM_HK",
            "china_pro2_no_TW_AM_HK.shp"
        ),
    ]

    for base in [csv_dir, cwd]:
        for name in candidate_names:
            candidates.append(base / name)

    deduped = []
    seen = set()
    for p in candidates:
        sp = str(p)
        if sp not in seen:
            deduped.append(p)
            seen.add(sp)

    for p in deduped:
        if p.exists():
            return str(p.resolve())

    raise FileNotFoundError(
        "未找到省级 shp，请检查路径。已尝试：\n" +
        "\n".join(str(p) for p in deduped)
    )


def choose_metric_work_crs(gdf):
    """最近匹配最好在投影坐标系下做，若原 CRS 是经纬度，则转到 EPSG:3857。"""
    if gdf.crs is None:
        raise ValueError("shp 缺少 CRS 信息，无法进行空间匹配。")

    try:
        is_geographic = bool(gdf.crs.is_geographic)
    except Exception:
        is_geographic = False

    return "EPSG:3857" if is_geographic else gdf.crs


def detect_shp_fields(province_gdf):
    code_col = detect_column(
        province_gdf,
        manual_col=None,
        candidates=["ADCODE99", "ADCODE", "adcode99", "adcode"],
        fuzzy_tokens=["adcode", "code"],
        label="省代码"
    )

    name_col = detect_column(
        province_gdf,
        manual_col=None,
        candidates=["NAME_EN_JX", "NAME_EN", "NAME", "PROVINCE", "Province"],
        fuzzy_tokens=["nameenjx", "nameen", "province", "name"],
        label="省名称"
    )

    return code_col, name_col


def nearest_join_fallback(points_unmatched, province_work):
    """当 sjoin_nearest 不可用时，使用空间索引做最近匹配兜底。"""
    idx_pairs, distances = province_work.sindex.nearest(
        points_unmatched.geometry,
        return_all=False,
        return_distance=True,
    )

    left_rows = points_unmatched.iloc[idx_pairs[0]].reset_index(drop=True)
    right_rows = province_work.iloc[idx_pairs[1]][["ADCODE99", "NAME_EN_JX"]].reset_index(drop=True)

    out = pd.concat([left_rows[["_row_id"]], right_rows], axis=1)
    out["distance_to_province"] = distances
    return out


def standardize_name_en_jx(province_gdf, original_name_field):
    """若 shp 缺少 NAME_EN_JX，则优先按 ADCODE99 自动补标准英文省名。"""
    province_gdf = province_gdf.copy()
    adc_num = pd.to_numeric(province_gdf["ADCODE99"], errors="coerce").astype("Int64")

    if original_name_field != "NAME_EN_JX":
        mapped = adc_num.map(PROVINCE_NAME_EN_FALLBACK)
        province_gdf["NAME_EN_JX"] = mapped.fillna(province_gdf["NAME_EN_JX"])

    return province_gdf


In [ ]:

# 主函数：匹配并原地覆盖 CSV

def add_province_fields_to_csv(
    input_csv,
    province_shp_path=None,
    lon_col=None,
    lat_col=None,
    backup_original=True,
    save_match_method_column=False,
    output_encoding=None,
):
    input_csv = str(Path(input_csv).expanduser().resolve())

    if not os.path.exists(input_csv):
        raise FileNotFoundError(f"未找到 CSV：{input_csv}")

    province_shp_path = resolve_province_shp(input_csv, province_shp_path)

    print("=" * 80)
    print("输入 CSV:", input_csv)
    print("省级 shp:", province_shp_path)
    print("=" * 80)

    # 1) 读取 CSV
    df, read_encoding = read_csv_with_fallback(input_csv)
    print(f"CSV 编码: {read_encoding}")
    print(f"CSV 行数 / 列数: {df.shape}")

    # 2) 自动识别经纬度列
    lon_candidates = [
        "Longitude", "longitude", "LONGITUDE", "lon", "Lon", "LON",
        "lng", "Lng", "LNG", "经度", "jingdu"
    ]
    lat_candidates = [
        "Latitude", "latitude", "LATITUDE", "lat", "Lat", "LAT",
        "纬度", "weidu"
    ]

    lon_col = detect_column(
        df, lon_col,
        candidates=lon_candidates,
        fuzzy_tokens=["longitude", "lon", "lng", "经度", "jingdu"],
        label="经度"
    )
    lat_col = detect_column(
        df, lat_col,
        candidates=lat_candidates,
        fuzzy_tokens=["latitude", "lat", "纬度", "weidu"],
        label="纬度"
    )

    print(f"识别到经度列: {lon_col}")
    print(f"识别到纬度列: {lat_col}")

    df = df.copy()
    df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")
    df[lat_col] = pd.to_numeric(df[lat_col], errors="coerce")

    invalid_range_mask = (
        df[lon_col].notna() & ((df[lon_col] < -180) | (df[lon_col] > 180))
    ) | (
        df[lat_col].notna() & ((df[lat_col] < -90) | (df[lat_col] > 90))
    )

    if invalid_range_mask.any():
        print(f"⚠️ 发现 {int(invalid_range_mask.sum())} 行超出经纬度范围，已按缺失坐标处理。")
        df.loc[invalid_range_mask, [lon_col, lat_col]] = np.nan

    df["_row_id"] = np.arange(len(df))
    valid_coord_mask = df[lon_col].notna() & df[lat_col].notna()

    print(f"有效经纬度点数: {int(valid_coord_mask.sum())} / {len(df)}")
    print(f"缺失经纬度点数: {int((~valid_coord_mask).sum())}")

    # 3) 读取省级 shp
    province_gdf = gpd.read_file(province_shp_path)
    code_src_col, name_src_col = detect_shp_fields(province_gdf)

    print("省界 CRS:", province_gdf.crs)
    print(f"省代码字段: {code_src_col}")
    print(f"省名称字段: {name_src_col}")

    province = province_gdf[[code_src_col, name_src_col, "geometry"]].copy()
    province = province.rename(columns={code_src_col: "ADCODE99", name_src_col: "NAME_EN_JX"})
    province = province[province.geometry.notna()].copy()
    province = standardize_name_en_jx(province, name_src_col)

    if name_src_col != "NAME_EN_JX":
        print(
            f"⚠️ shp 中没有 NAME_EN_JX 字段，"
            f"已优先用 ADCODE99 自动补标准英文省名；必要时退回 {name_src_col}。"
        )

    # 4) 空间匹配：先 within，再 nearest
    if valid_coord_mask.sum() == 0:
        joined_final = pd.DataFrame(
            columns=["_row_id", "ADCODE99", "NAME_EN_JX", "province_match_method", "distance_to_province"]
        )
        print("没有可用于空间匹配的有效经纬度。")
    else:
        points_df = df.loc[valid_coord_mask, ["_row_id", lon_col, lat_col]].copy()
        points_gdf = gpd.GeoDataFrame(
            points_df,
            geometry=gpd.points_from_xy(points_df[lon_col], points_df[lat_col]),
            crs="EPSG:4326",
        )

        work_crs = choose_metric_work_crs(province)
        province_work = province.to_crs(work_crs) if str(province.crs) != str(work_crs) else province.copy()
        points_work = points_gdf.to_crs(work_crs)

        # 4.1 严格面内匹配
        joined = gpd.sjoin(
            points_work,
            province_work,
            how="left",
            predicate="within"
        )

        joined = (
            joined.sort_values("_row_id")
                  .drop_duplicates(subset="_row_id", keep="first")
                  .set_index("_row_id")
        )

        joined["province_match_method"] = np.where(joined["ADCODE99"].notna(), "within", "unmatched")
        joined["distance_to_province"] = np.where(joined["ADCODE99"].notna(), 0.0, np.nan)

        unmatched_ids = joined.index[joined["ADCODE99"].isna()]
        print("within 未匹配到的点数:", int(len(unmatched_ids)))

        # 4.2 对未匹配点做最近匹配
        if len(unmatched_ids) > 0:
            points_unmatched = points_work[points_work["_row_id"].isin(unmatched_ids)].copy()

            try:
                nearest = gpd.sjoin_nearest(
                    points_unmatched,
                    province_work,
                    how="left",
                    distance_col="distance_to_province"
                )
                nearest = (
                    nearest.sort_values(["_row_id", "distance_to_province"])
                           .drop_duplicates(subset="_row_id", keep="first")
                           .set_index("_row_id")
                )
            except Exception as e:
                print("⚠️ sjoin_nearest 执行失败，改用 sindex.nearest 兜底。")
                print("错误信息：", e)
                nearest = nearest_join_fallback(points_unmatched, province_work).set_index("_row_id")

            fill_ids = nearest.index[nearest["ADCODE99"].notna()]

            joined.loc[fill_ids, "ADCODE99"] = nearest.loc[fill_ids, "ADCODE99"]
            joined.loc[fill_ids, "NAME_EN_JX"] = nearest.loc[fill_ids, "NAME_EN_JX"]
            joined.loc[fill_ids, "distance_to_province"] = nearest.loc[fill_ids, "distance_to_province"].values
            joined.loc[fill_ids, "province_match_method"] = np.where(
                nearest.loc[fill_ids, "distance_to_province"].fillna(0).eq(0),
                "nearest_0m",
                "nearest"
            )

        joined_final = joined.reset_index()[[
            "_row_id", "ADCODE99", "NAME_EN_JX", "province_match_method", "distance_to_province"
        ]]

        print("匹配方式统计：")
        display(joined_final["province_match_method"].value_counts(dropna=False))
        print("仍未匹配到省份的点数:", int(joined_final["ADCODE99"].isna().sum()))

    # 5) 回填结果
    result_df = df.copy()

    # 强制覆盖 / 新建目标列
    result_df["ADCODE99"] = pd.NA
    result_df["NAME_EN_JX"] = pd.NA

    if save_match_method_column:
        result_df["province_match_method"] = "missing_coordinates"

    if len(joined_final) > 0:
        result_df.loc[joined_final["_row_id"], "ADCODE99"] = joined_final["ADCODE99"].values
        result_df.loc[joined_final["_row_id"], "NAME_EN_JX"] = joined_final["NAME_EN_JX"].values

        if save_match_method_column:
            result_df.loc[joined_final["_row_id"], "province_match_method"] = (
                joined_final["province_match_method"].fillna("unmatched").values
            )

    if save_match_method_column:
        result_df.loc[~valid_coord_mask, "province_match_method"] = "missing_coordinates"

    # 尽量转整数型
    try:
        result_df["ADCODE99"] = pd.to_numeric(result_df["ADCODE99"], errors="coerce").astype("Int64")
    except Exception:
        pass

    # 删除中间列
    result_df = result_df.drop(columns=["_row_id"])

    # 6) 原地覆盖 CSV（先备份，再临时文件替换）
    if backup_original:
        backup_path = input_csv + ".bak"
        shutil.copy2(input_csv, backup_path)
        print("已备份原文件：", backup_path)

    tmp_output = input_csv + ".tmp"
    write_encoding = output_encoding or read_encoding
    if write_encoding == "latin1":
        write_encoding = "utf-8-sig"

    result_df.to_csv(tmp_output, index=False, encoding=write_encoding)
    os.replace(tmp_output, input_csv)

    print("✅ 已覆盖原 CSV：", input_csv)
    print("总行数:", len(result_df))
    print("成功匹配行数:", int(result_df["ADCODE99"].notna().sum()))
    print("未匹配行数:", int(result_df["ADCODE99"].isna().sum()))

    return result_df


In [ ]:

# 执行

result_df = add_province_fields_to_csv(
    input_csv=input_csv,
    province_shp_path=province_no_TW_AM_HK_shp,
    lon_col=lon_col,
    lat_col=lat_col,
    backup_original=backup_original,
    save_match_method_column=save_match_method_column,
    output_encoding=output_encoding,
)

print("\n处理完成，预览前 5 行：")
display(result_df.head())


In [ ]:

# 可选：查看需要人工复核的样本
# 仅当 save_match_method_column = True 时，这一格才会显示匹配方式。

if "province_match_method" not in result_df.columns:
    print("当前 save_match_method_column = False，未写出 province_match_method 列。")
else:
    review_df = result_df[result_df["province_match_method"] != "within"].copy()
    if len(review_df) == 0:
        print("所有有效坐标点都通过 within 直接匹配完成。")
    else:
        print(f"需要复核的样本数: {len(review_df)}")
        show_cols = [c for c in [lon_col, lat_col, "ADCODE99", "NAME_EN_JX", "province_match_method"] if c in review_df.columns]
        display(review_df[show_cols].head(20))
